# Instamatic - installation and configuration
### Prerequisites
This tutorial assumes that you have a basic understanding of python, python virtual environments, and a virtual environment with Python 3.10+ that includes a jupyter notebook that allows to read this file. In case any of these is not the case, please follow the lecture by Magnus Nord on setting up Python or ask any of the available staff.

Please also note that Instamatic was never tested on a non-windows platform. Selected functionality may be limited or non-functional. For example, the API revealed by the microscopes and thus the objects utilized by Instamatic rely on a COM architecture that is strictly windows-specific. Without these libraries in place, Instamatic will certainly be unable to interface any microscope, but might still work with the emulator for presentation purposes.

### Setting up the environment
By this point, you should have a virtual environment with instamatic set up and ready to go. The instructions to set it up should have looked something along the following lines:

    conda create --name instamatic-venv
    conda activate instamatic-venv
    conda install git python jupyterlab -c conda-forge
    git clone https://github.com/instamatic-dev/instamatic/
    pip install -e ./instamatic
    git clone https://github.com/instamatic-dev/instamatic-TEM-emulator/
    cd instamatic-TEM-emulator
    pip install -r requirements.txt

This set of commands should create a conda virtual environment called `instamatic-venv` that includes an editable (`-e`) version of instamatic as well as the repository with the TEM emulator.

### Configuring instamatic
Firstly, let's focus on configuring and getting the very basic functionality of Instamatic running. In order to memorize configurations and settings between sessions, Instamatic accesses and stores information inside the local environment variable `AppData`. On Windows machines, this can be typically accessed by typing `%Appdata%` into the address bar. Unix-based computer are likely not to define `$AppData` by default, so the user should do so themselves in a location of choice. Because `AppData` is typically shared by the system, even if the user has multiple instamatic installations, the configuration is always shared between them. If you ever worked with Instamatic of your computer, it is therefore suggested to back up and then restore your `%AppData%\Instamatic` directory.

The location of Instamatic config in `%AppData%\Instamatic` will be absent after installation alone. In order to create the first draft of configuration, attempt to run instamatic once by calling `instamatic` in the command line (inside the virtual environment). This likely won't start the program, but instead will build the necessary file structure.

After the configuration is created, navigate to `%Appdata%\Instamatic` and open `settings.yaml`. Instamatic uses predominantly yaml file to store its settings, configuration, calibration etc. `settings.yaml` stores the very basic information about the expected server structure and the IP addresses that clients and servers should expect. Also note that over the year, Instamatic accumulated a lot of settings that are used only in specific occasions. For the purpose of basic testing you only need to make sure that the following six fields are set properly:

    microscope: simulate
    camera: simulate
    calibration: simulate
    simulate: False
    use_tem_server: False
    use_cam_server: False

These six settings inform Instamatic that it should not expect but rather simulate both the camera and a microscope. The basic camera and microscope simulations are painfully simplistic; for example, simulated image simply continuously streams an image composed of random integers, without any regards for the microscope. This is ,of course, scientifically negligent, but sufficient to assert a correct working of the program. For the time being do not worry about other setting, and after saving this configuration enter your conda environment and again attempt to start instamatic by calling `instamatic`. If you are successful, now you should see a GUI with a live view on the left side, I/O panel on the top-right, and multiple tabs with various experimental modules on the bottom-right, something like this:

![Instamatic GUI Image](https://instamatic.readthedocs.io/en/latest/images/gui_main.png)

This is a good point to briefly explore the Instamatic GUI or troubleshoot any basic issues. If you are interested, feel free to learn more about Instamatic config system as described in the [documentation](https://instamatic.readthedocs.io/en/latest/).

### Testing
Once you are done with the basic configuration of "simulated" setup where the TEM and camera are both basically non-functional dummy objects, it is a good idea to attempt the first series of tests. As a python package, instamatic does not come with the tests; therefore, if you installed instamatic from PyPI using `pip install instamatic`, you can skip this section. However, if you install the entire package directly from git, the tests come inside the repository. To run the tests, assert that your virtual environment has `pytest`, navigate to `instamatic\tests\` and run `pytest`. The tests may take up to a few minutes, but you should see that all the tests either run correctly or failed as expected (`xfailed`) because they were designed to test specific experimental features impossible to recreate on a simulator.

    ==================== test session starts ====================
    platform win32 -- Python 3.13.2, pytest-8.3.4, pluggy-1.5.0
    rootdir: C:\Users\tchon\PycharmProjects\instamatic
    configfile: pyproject.toml
    plugins: anyio-4.12.0
    collected 116 items

    test_camera.py ...xx..x...xx.xx                        [ 13%]
    test_collections.py ..........                         [ 22%]
    test_ctrl.py ...........                               [ 31%]
    test_deprecated.py .....                               [ 36%]
    test_experiments.py xx.....                            [ 42%]
    test_formats.py ................                       [ 56%]
    test_grid_mapping.py .                                 [ 56%]
    test_merlin_io.py .                                    [ 57%]
    test_pets_input.py .....                               [ 62%]
    test_simulation\test_crystal.py ..........             [ 70%]
    test_simulation\test_grid.py ..xx                      [ 74%]
    test_simulation\test_sample.py ..x.x                   [ 78%]
    test_simulation\test_stage.py ..xx                     [ 81%]
    test_tools.py .......                                  [ 87%]
    test_utils.py ..............                           [100%]

    ===================== warnings summary ======================
    tests/test_simulation/test_grid.py::test_get_array
      C:\...\instamatic\simulation\grid.py:69: NotImplementedWarning: Center mark is not implemented yet
    warnings.warn('Center mark is not implemented yet', NotImplementedWarning)

    -- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
    ======== 101 passed, 15 xfailed, 1 warning in 31.14s ========

### Connecting to a microscope
Once we confirm the calibration directory is created and read as expected, and then assert the basic functionality of the code, it is high time to connect Instamatic to the microscope. Unfortunately, Instamatic as a program was never designed to allow multiple connections to one server (microscope/camera), and there are not enough microscopes in the local network for every participant to connect to. Therefore, during the workshop we will explore the server-client connectivity of Instamatic using a local TEM emulator.

[Instamatic-TEM-emulator](https://github.com/instamatic-dev/instamatic-TEM-emulator) is a stand-alone TEM emulator designed as a real microscope/camera "replacement" to be used with instamatic testing/development. The part of the emulator responsible for generating an image was designed by Viljar Femoen, a former PhD student at Stockholm University. The "microscope" handled by the emulator is the same as the one built in Instamatic. In fact, it is imported from Instamatic at the start-up and, as such, any changes done to Instamatic simulated TEM will be reflected in the emulator upon its restart. In contrast to the simulated camera that produces noise only, the "emulated" camera accesses the current state of the virtual TEM and generates a simple visualisation based on the entirety of observed image. The current version of the emulator does not simulate beam at all: simply everything in the correct field of view contributes to diffraction.

Before starting the emulator we need to prepare some configuration files for it. We will discuss these later - for the time being, navigate to `%AppData%/instamatic/config/microscope`, make a copy of `simulate.yaml`, and name it `emulator.yaml`. Similarly, go to `%AppData%/instamatic/config/camera`, make a copy of `simulate.yaml`, and name it `emulator.yaml`

In order to start the emulator, open another terminal window and enter the `instamatic-venv` virtual environment dedicated to instamatic. Navigate to `instamatic-TEM-emulator/src/instamatic-tem-emulator` and start the server using `python start_server.py`. If you want to speed this process up for yourself, you can modify and use the `start_server.bat` file. After you start the server, the process running it should output some basic debug information and await connection to Instamatic:

    Config directory: C:\Users\tchon\AppData\Roaming\instamatic\config
    C:\...\instamatic-TEM-emulator\src\instamatic-tem-emulator\simulation\stage.py:38: NotImplementedWarning: Tilting is not fully implemented yet
      self.set_position(0, 0, 0, 0, 0)

If you are interested about more that is going on in the background, you can check out the log that should be present in `%appdata%\instamatic\logs\instamatic_TEM_emulator_2026-06-02.log`:

    2026-06-02 15:59:46,444 root: INFO     Emulator starting
    2026-06-02 15:59:46,445 tem : INFO     Started simulate microscope server thread
    2026-06-02 15:59:46,466 cam : INFO     Started camera listener thread
    2026-06-02 15:59:46,466 tem : INFO     Started microscope listener thread

Now that the emulator is running, you would think you can simply start and connect to it using Instamatic. To test that theory, start the program again by calling `instamatic` in the command line. Unfortunately, what you should see is exactly the same window. You can try requesting some operations from the emulated TEM, but there should be no response from the emulator - after all, we have not configured Instamatic to know where to connect yet!

In order to properly configure both instamatic and the TEM emulator, navigate again to the config in `%appdata%\instamatic\settings.yaml` and assert the content at the top aligns with to the following:

    microscope: simulate
    camera: simulate
    simulate: False

    use_tem_server: True
    tem_server_host: 'localhost'
    tem_server_port: 5000
    tem_require_admin: False
    tem_communication_protocol: 'pickle'

    use_cam_server: True
    cam_server_host: 'localhost'
    cam_server_port: 5001
    cam_use_shared_memory: True

Port numbers do not have to exactly match these shown above, they just need to be free, not used for communication by any other program.

With these changes in place, re-start instamatic-TEM-emulator and then start Instamatic. Now, instamatic should preview a simplistic view of the grid and allow preview with for reasonable feedback. For example, the emulator allows to run the control panel or several kinds of experiments (RED, cRED, FastADT) and see the feedback. If you want, (upon closing the GUI) you can also repeat the tests, which now will run on the emulator rather than in the built-in simulator, albeit because the emulator uses the same simulated TEM object under the hood, the overall results should be the same.

### Configuring the TEM and camera (advanced)
The default settings used by the simulated microscope and camera should be sufficient to run a simple presentation using emulator, but they will definitely need to be adapted to a real machine or detector. In order to be informed about all the peculiarities of any given hardware, the program relies not only on the `settings.yaml` that we have already explored, but also all the other files present in `%appdata%\config`:
* `settings.yaml` stores links to other calibration files information about basic functionality and server/client architecture.
* `default.yaml` stores default values for some grid montage / machine learning routines.
* `microscope` stores multiple microscope calibration files; only the one specified on top of `settings.yaml` is used; the microscope settings include programmatic interface to be used, magnification ranges, and wavelength.
* `camera` stores multiple camera calibration files; only the one specified on top of `settings.yaml` is used; these include parameters such as default and possible binning, exposure, dimensions, pixel size, or any other `detector_config` passed directly to camera.
* `calibrate` stores multiple tem-cam calibration files (+ some other files); only the one specified on top of `settings.yaml` is used; calibration files can specify pre-calibrated pixel size and transformation matrix associated with every magnification.

In order to explore the meaning behind individual parameters present in each of the files, please refer to the [documentation](https://instamatic.readthedocs.io/en/latest/).

Programmatic interfaces available in Instamatic can be explored by investigating the source code at `instamatic/microscope/interface` and `instamatic/camera`. They correspond to one of the more complicated and older parts of the code. Ideally, when configuring a new setup, you would prefer to include and re-use one of the existing implementations because adding a new camera / microscope is not straightforward / very hard, respectively. As of now, the easiest way to add a new implementation directly to Instamatic, forking (copying) the repository, writing it in a new file, and optimally contributing it back to the main repository via a git merge/pull request. However, interfaces can be also written as completely external programs, as is the case with (`instamatic_tecnai_server`)[https://github.com/instamatic-dev/instamatic-tecnai-server] which demands Python 3.4 and thus must be developed independently of the main branch.